# 06｜Flask Web API

提供首頁、影像串流、狀態、上傳、控制、紀錄下載與 LINE 測試端點。


In [ ]:
@app.route('/')
def index():
    return render_template_string(HTML_PAGE)

@app.route('/video_feed')
def video_feed():
    def generate():
        while True:
            frame = state.get_frame()
            if frame is None:
                frame = BLACK_FRAME_JPEG
            yield (b'--frame\r\n'
                   b'Content-Type: image/jpeg\r\n\r\n' + frame + b'\r\n')
            time.sleep(0.03)
    return Response(generate(), mimetype='multipart/x-mixed-replace; boundary=frame')

@app.route('/stats')
def stats():
    text, warning, paused, running, log_ready = state.get_stats()
    return jsonify({"stats": text, "warning": warning, "running": running,
                     "paused": paused, "log_ready": log_ready})

@app.route('/upload', methods=['POST'])
def upload():
    if state.running:
        return jsonify({"ok": False, "msg": "處理中無法上傳新檔案，請先按停止"})

    file = request.files.get('video_file')
    if not file or file.filename == '':
        return jsonify({"ok": False, "msg": "請選擇影片檔案"})

    file.save(UPLOAD_PATH)
    state.file_uploaded = True

    first_frame_jpeg = extract_first_frame_jpeg(UPLOAD_PATH)
    if first_frame_jpeg is not None:
        state.update_frame(first_frame_jpeg)
    else:
        state.clear_frame()

    state.update_stats("新影片已上傳，尚未開始處理", False)
    state.log_ready = False

    return jsonify({"ok": True, "msg": "上傳完成"})

@app.route('/start', methods=['POST'])
def start():
    if state.thread is not None and state.thread.is_alive():
        state.running = False
        state.thread.join(timeout=10)

    source_type = request.form.get('source_type')
    conf_threshold = float(request.form.get('conf_threshold', 0.25))
    warning_ratio = float(request.form.get('warning_ratio', 0.6))
    warning_window_sec = float(request.form.get('warning_window_sec', 5.0))
    min_ratio_of_window = float(request.form.get('min_ratio_of_window', 0.3))

    state.line_enabled = request.form.get('line_enabled', 'false') == 'true'
    state.line_token = request.form.get('line_token', '').strip()

    if state.line_enabled and not state.line_token:
        return jsonify({"ok": False, "msg": "已勾選LINE通知，但尚未輸入CHANNEL_ACCESS_TOKEN"})

    if source_type == 'upload':
        if not state.file_uploaded or not os.path.exists(UPLOAD_PATH):
            return jsonify({"ok": False, "msg": "請先上傳影片檔案"})
        source = UPLOAD_PATH
        is_stream = False
    else:
        raw_stream = request.form.get('stream_url', '').strip()
        if not raw_stream:
            return jsonify({"ok": False, "msg": "請輸入串流網址或攝影機編號"})
        source = resolve_source(raw_stream, True)
        is_stream = True

    model.predictor = None

    state.running = True
    state.paused = False
    state.log_ready = False
    state.update_stats("啟動中...", False)

    state.thread = threading.Thread(
        target=processing_worker,
        args=(source, is_stream, conf_threshold, warning_ratio,
              warning_window_sec, min_ratio_of_window),
        daemon=True
    )
    state.thread.start()
    return jsonify({"ok": True, "msg": "已開始處理"})

@app.route('/stop', methods=['POST'])
def stop():
    state.running = False
    state.paused = False
    if state.thread is not None and state.thread.is_alive():
        state.thread.join(timeout=10)
    return jsonify({"ok": True, "msg": "已停止"})

@app.route('/pause', methods=['POST'])
def pause():
    if not state.running:
        return jsonify({"ok": False, "msg": "目前沒有正在執行的任務"})
    state.paused = True
    return jsonify({"ok": True, "msg": "已暫停"})

@app.route('/resume', methods=['POST'])
def resume():
    if not state.running:
        return jsonify({"ok": False, "msg": "目前沒有正在執行的任務"})
    state.paused = False
    return jsonify({"ok": True, "msg": "已繼續"})

@app.route('/download_log')
def download_log():
    if not os.path.exists(LOG_PATH):
        return jsonify({"ok": False, "msg": "尚無可下載的log"}), 404
    return send_file(LOG_PATH, as_attachment=True,
                      download_name=f"detection_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")

@app.route('/test_line', methods=['POST'])
def test_line():
    token = request.form.get('line_token', '').strip()
    if not token:
        return jsonify({"ok": False, "msg": "請先輸入CHANNEL_ACCESS_TOKEN"})
    try:
        line_bot_api = LineBotApi(token)
        line_bot_api.broadcast(TextSendMessage(text="✅ 溺水偵測系統：測試訊息，LINE通知功能已成功連接。"))
        return jsonify({"ok": True, "msg": "測試訊息已送出，請確認LINE官方帳號是否收到"})
    except Exception as e:
        return jsonify({"ok": False, "msg": f"發送失敗: {e}"})
